# НИС «Основы анализа данных в Python»

*Алла Тамбовцева*

## Практикум 4. Датафреймы `pandas`: добавление новых столбцов

In [1]:
import pandas as pd

## Часть 1: работа с данными по Олимпийским играм

В файле `games.csv` сохранены результаты Олимпийских игр в разные годы. Одна строка таблицы соответствует одной команде – одной стране – в определенный год. 

Ссылка на файл: https://github.com/allatambov/PyDat25/blob/main/games.csv

Показатели в файле:

* `year`: год;
* `host_country`: страна, принимающая Игры;
* `host_city`: город, где проходили Игры;
* `athletes`: число спортсменов на Играх;
* `teams`: число команд-участников;
* `competitions`: число соревнований в рамках Игр;
* `country`: команда, страна-участник Игр;
* `gold`: количество золотых медалей, полученных командой страны на Играх;
* `silver`: количество серебряных медалей, полученных командой страны на Играх;
* `bronze`: количество бронзовых медалей, полученных командой страны на Играх.

Загрузите данные из файла `games.csv` и сохраните их в датафрейм `games`. Проверьте, есть ли в датафрейме строки с пропущенными значениями.

In [2]:
games = pd.read_csv("games.csv")
games.head()

,year,games_type,host_country,host_city,athletes,teams,competitions,country,gold,silver,bronze
0,2022,Winter,China,Beijing,2834,91,109,Australia,1,2,1
1,2022,Winter,China,Beijing,2834,91,109,Austria,7,7,4
2,2022,Winter,China,Beijing,2834,91,109,Belarus,0,2,0
3,2022,Winter,China,Beijing,2834,91,109,Belgium,1,0,1
4,2022,Winter,China,Beijing,2834,91,109,Canada,4,8,14


In [3]:
# пропусков нет

games.isna().sum()

year            0
games_type      0
host_country    0
host_city       0
athletes        0
teams           0
competitions    0
country         0
gold            0
silver          0
bronze          0
dtype: int64

### Задача 1

Добавьте в `games` столбец `total` с общим числом золотых, серебряных и бронзовых медалей. На каких Играх было получено самое большое количество медалей? Укажите год и название принимающей страны. 

In [4]:
games["total"] = games["gold"] + games["silver"] + games["bronze"]
games.head()

,year,games_type,host_country,host_city,athletes,teams,competitions,country,gold,silver,bronze,total
0,2022,Winter,China,Beijing,2834,91,109,Australia,1,2,1,4
1,2022,Winter,China,Beijing,2834,91,109,Austria,7,7,4,18
2,2022,Winter,China,Beijing,2834,91,109,Belarus,0,2,0,2
3,2022,Winter,China,Beijing,2834,91,109,Belgium,1,0,1,2
4,2022,Winter,China,Beijing,2834,91,109,Canada,4,8,14,26


In [5]:
# выбираем строки с максимальным total,
# смотрим на year и host_country

res = games[games["total"] == games["total"].max()]
res

,year,games_type,host_country,host_city,athletes,teams,competitions,country,gold,silver,bronze,total
1747,1904,Summer,United States,St. Louis,651,12,95,United States of America,76,78,77,231


Если хотим получить отдельно год и отдельно принимающую страну, в методе `.loc[]` можем выбрать номер строки и название столбца, на пересечении которых находится нужная нам ячейка:

In [6]:
# выше видно, что индекс строки 1747, это номер в исходном датафрейме games

print(res.loc[1747, "year"])
print(res.loc[1747, "host_country"])

1904
United States


Если сначала выберем столбец, это будет объект `pandas Series`, последовательность значений с индексами:

In [7]:
res["year"]

1747    1904
Name: year, dtype: int64

Поэтому из него нужно будет забрать значения – атрибут .`values`:

In [8]:
res["year"].values

array([1904])

А из полученного массива `numpy array` – единственный элемент с индексом 0:

In [9]:
print(res["year"].values[0])
print(res["host_country"].values[0])

1904
United States


**Дополнительно:** так как первый способ с `.loc` требует известного номера строки, который нам заранее неизвестен (если мы все автоматизируем, мы не предполагаем, что видим промежуточные результаты глазами), а второй – довольно громоздкий, можно рассмотреть еще два варианта. 

In [10]:
# не сохраняя изменений в res, «обнуляем» индексы строк, 
# у строки теперь номер гарантированно 0, а не 1747

print(res.reset_index().loc[0, "year"])
print(res.reset_index().loc[0, "host_country"])

1904
United States


In [11]:
# узнаем номер строки, где достигается максимум по total,
# подставляем его в loc[]

print(res.loc[res["total"].idxmax(), "year"])
print(res.loc[res["total"].idxmax(), "host_country"])

1904
United States


### Задача 2

Не создавая новых столбцов, вычислите, сколько команд получило золотых медалей больше, чем серебряных и бронзовых вместе взятых.

In [12]:
(games["gold"] > games["silver"] + games["bronze"]).sum()

193

### Задача 3

Выведите таблицу частот для типа Игр (`games_type`). Посмотрите на значения и добавьте в `games`:

* бинарный столбец `summer`, где 1 соответствует летним Играм, а 0 – зимним;
* столбец `season`, где 1 соответствует летним играм, а 2 – зимним.

In [13]:
games["games_type"].value_counts()

Summer    1339
Winter     442
Name: games_type, dtype: int64

In [14]:
# бинарный столбец – переделываем True/False в int

games["summer"] = (games["games_type"] == "Summer").astype(int)
games.head(3)

,year,games_type,host_country,host_city,athletes,teams,competitions,country,gold,silver,bronze,total,summer
0,2022,Winter,China,Beijing,2834,91,109,Australia,1,2,1,4,0
1,2022,Winter,China,Beijing,2834,91,109,Austria,7,7,4,18,0
2,2022,Winter,China,Beijing,2834,91,109,Belarus,0,2,0,2,0


In [15]:
# столбец из 1 и 2 – через lambda-функцию

games["season"] = games["games_type"].apply(lambda x: 1 if x == "Summer" else 2)
games.head(3)

,year,games_type,host_country,host_city,athletes,teams,competitions,country,gold,silver,bronze,total,summer,season
0,2022,Winter,China,Beijing,2834,91,109,Australia,1,2,1,4,0,2
1,2022,Winter,China,Beijing,2834,91,109,Austria,7,7,4,18,0,2
2,2022,Winter,China,Beijing,2834,91,109,Belarus,0,2,0,2,0,2


### Задача 4

Добавьте в `games` бинарный столбец `matching`, где 1 соответствует наблюдениям, где выступающая страна совпадает с принимающей страной, а 0 – всем остальным наблюдениям. Другими словами, значения 1 должны стоять в тех случаях, когда команда играла «дома». Вычислите количество и долю таких случаев.

In [16]:
games["matching"] = (games["country"] == games["host_country"]).astype(int)

In [17]:
# число случаев – сумма по набору из 0 и 1
# доля случаев – среднее по набору из 0 и 1

print(games["matching"].sum())
print(games["matching"].mean())

39
0.021897810218978103


## Часть 2: работа с новостями

В файле `nplus1_2025.csv` сохранены характеристики новостей науки с сайта [N+1](https://nplus1.ru/):

* `title`: заголовок новости;
* `author`: автор новости;
* `date`: дата публикации новости;
* `diffc`: сложность новости;
* `rubrics`: рубрики новости;
* `text`: текст новости.

Файл не совсем маленький (~10 Мб), если есть проблемы со скачиванием или загрузкой в Google Colab, его загрузить можно сразу по ссылке (https://raw.githubusercontent.com/allatambov/PyDat25/refs/heads/main/nplus1_2025.csv), скопировав ее в `read_csv()`.

Загрузите данные из файла `nplus1_2025.csv` и сохраните их в датафрейм `news`. Удалите столбец `Unnamed: 0`, используя метод `.drop()`.

In [18]:
news = pd.read_csv("nplus1_2025.csv")
news.head()

,Unnamed: 0,title,author,date,diffc,rubrics,text
0,0,Температуру ядра Земли уточнили с помощью лазе...,Егор Конюхов,2025-01-03,2.9,"Физика, Геология",Физики уточнили температуру внутренней границы...
1,1,Большинство людей эпохи мезолита из Оленеостро...,Михаил Подрезов,2025-01-03,3.1,"Археология, Антропология",Ученые проанализировали изотопный состав строн...
2,2,Физики разобрались в состоянии покоя ткани. Он...,Егор Конюхов,2025-01-03,2.3,Физика,"Группа физиков выяснила, что при отсутствии вн..."
3,3,Виноградины усилили микроволновое поле и помог...,Егор Конюхов,2025-01-06,4.1,Физика,Физики использовали две виноградины для усилен...
4,4,Прием глюкокортикоидов при беременности повыси...,Олег Лищук,2025-01-06,1.2,"Медицина, Психология",Кристина Лаугесен (Kristina Laugesen) из Орхус...


In [19]:
# удвляем столбец + сохраняем изменения через inplace = True

news.drop(columns = ["Unnamed: 0"], inplace = True)

### Задача 1

Определите максимальное значение сложности статьи. Добавьте в `news` столбец с относительной сложностью статьи – сложность статьи в процентах от самой сложной, округленной до сотых.

In [20]:
m = news["diffc"].max()
news["rel_diffc"] = (news["diffc"] / m * 100).round(2)

news.head()

,title,author,date,diffc,rubrics,text,rel_diffc
0,Температуру ядра Земли уточнили с помощью лазе...,Егор Конюхов,2025-01-03,2.9,"Физика, Геология",Физики уточнили температуру внутренней границы...,31.87
1,Большинство людей эпохи мезолита из Оленеостро...,Михаил Подрезов,2025-01-03,3.1,"Археология, Антропология",Ученые проанализировали изотопный состав строн...,34.07
2,Физики разобрались в состоянии покоя ткани. Он...,Егор Конюхов,2025-01-03,2.3,Физика,"Группа физиков выяснила, что при отсутствии вн...",25.27
3,Виноградины усилили микроволновое поле и помог...,Егор Конюхов,2025-01-06,4.1,Физика,Физики использовали две виноградины для усилен...,45.05
4,Прием глюкокортикоидов при беременности повыси...,Олег Лищук,2025-01-06,1.2,"Медицина, Психология",Кристина Лаугесен (Kristina Laugesen) из Орхус...,13.19


### Задача 2

Добавьте в `news` столбец `author_name` и столбец `author_surname`, используя lambda-функции и метод `.apply()`.

In [21]:
# пишем lambda-функцию с методом .split()
# сначала смотрим на результат разбиения

news["author"].apply(lambda x: x.split())

0           [Егор, Конюхов]
1        [Михаил, Подрезов]
2           [Егор, Конюхов]
3           [Егор, Конюхов]
4             [Олег, Лищук]
               ...         
1772    [Катерина, Петрова]
1773    [Александр, Войтюк]
1774     [Михаил, Подрезов]
1775     [Михаил, Подрезов]
1776      [Сергей, Коленов]
Name: author, Length: 1777, dtype: object

In [22]:
# забираем из полученных список первый и второй элемент отдельно

news["author_name"] = news["author"].apply(lambda x: x.split()[0])
news["author_surname"] = news["author"].apply(lambda x: x.split()[1])

news.head()

,title,author,date,diffc,rubrics,text,rel_diffc,author_name,author_surname
0,Температуру ядра Земли уточнили с помощью лазе...,Егор Конюхов,2025-01-03,2.9,"Физика, Геология",Физики уточнили температуру внутренней границы...,31.87,Егор,Конюхов
1,Большинство людей эпохи мезолита из Оленеостро...,Михаил Подрезов,2025-01-03,3.1,"Археология, Антропология",Ученые проанализировали изотопный состав строн...,34.07,Михаил,Подрезов
2,Физики разобрались в состоянии покоя ткани. Он...,Егор Конюхов,2025-01-03,2.3,Физика,"Группа физиков выяснила, что при отсутствии вн...",25.27,Егор,Конюхов
3,Виноградины усилили микроволновое поле и помог...,Егор Конюхов,2025-01-06,4.1,Физика,Физики использовали две виноградины для усилен...,45.05,Егор,Конюхов
4,Прием глюкокортикоидов при беременности повыси...,Олег Лищук,2025-01-06,1.2,"Медицина, Психология",Кристина Лаугесен (Kristina Laugesen) из Орхус...,13.19,Олег,Лищук


### Задача 3

Выведите таблицу частот для столбца со сложностью новости. Добавьте разбиение на три группы с помощью аргумента `bins`. 

Напишите функцию `set_group()`, которая принимает на вход число от 1 до 10, и возвращает:

* значение `low`, если значение сложности принадлежит первому интервалу в полученном ранее разбиении;
* значение `medium`, если значение сложности принадлежит второму интервалу в полученном разбиении;
* значение `high`, если значение сложности принадлежит третьему интервалу в полученном разбиении.

Используя функцию `set_group()`, добавьте в `news` новый столбец `category` со степенью сложности новости.

In [23]:
# уникальных значений много – столбец из дробных чисел,
# поэтому объединяем их в 3 группы

news["diffc"].value_counts(bins = 3)

(1.0910000000000002, 3.767]    1416
(3.767, 6.433]                  325
(6.433, 9.1]                     36
Name: diffc, dtype: int64

In [24]:
def set_group(x):
    if x <= 3.767:
        y = "low"
    elif x <= 6.433:
        y = "medium"
    else:
        y = "high"
    return y

In [25]:
# применям написанную функцию

news["category"] = news["diffc"].apply(set_group)
news.head()

,title,author,date,diffc,rubrics,text,rel_diffc,author_name,author_surname,category
0,Температуру ядра Земли уточнили с помощью лазе...,Егор Конюхов,2025-01-03,2.9,"Физика, Геология",Физики уточнили температуру внутренней границы...,31.87,Егор,Конюхов,low
1,Большинство людей эпохи мезолита из Оленеостро...,Михаил Подрезов,2025-01-03,3.1,"Археология, Антропология",Ученые проанализировали изотопный состав строн...,34.07,Михаил,Подрезов,low
2,Физики разобрались в состоянии покоя ткани. Он...,Егор Конюхов,2025-01-03,2.3,Физика,"Группа физиков выяснила, что при отсутствии вн...",25.27,Егор,Конюхов,low
3,Виноградины усилили микроволновое поле и помог...,Егор Конюхов,2025-01-06,4.1,Физика,Физики использовали две виноградины для усилен...,45.05,Егор,Конюхов,medium
4,Прием глюкокортикоидов при беременности повыси...,Олег Лищук,2025-01-06,1.2,"Медицина, Психология",Кристина Лаугесен (Kristina Laugesen) из Орхус...,13.19,Олег,Лищук,low


**Дополнительно:** строго говоря, наша функция `set_group()` не очень хорошая – она не предусматривает наличие пропущенных или некорректных значений, если бы они были, применить такую функцию через `.apply()` не получилось бы. Решение простое – вместо `else` написать `elif x <= 9.1`, а в `else` поставить `y = None` или `y = np.nan` (если импортирован `numpy`), чтобы на пропуск и некорректное числовое значение возвращалось пропущенное значение.